# Feature File Inspection

This notebook inspects the feature pickle files to understand their structure, granularity, and consistency.
We suspect that some files might not be at the stock-day level, leading to merge issues.


In [22]:
import pandas as pd
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Define path
DATA_DIR = Path(r"c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv")
FEATURES_DIR = DATA_DIR / "features_mlcrowd"

print(f"Features Directory: {FEATURES_DIR}")


Features Directory: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\features_mlcrowd


In [23]:
# List all feature files
feature_files = sorted([f for f in FEATURES_DIR.glob("*.pkl") if "master" not in f.name])

print(f"Found {len(feature_files)} feature files:")
for f in feature_files:
    print(f" - {f.name}")


Found 3 feature files:
 - features_01_basic_sentiment.pkl
 - features_02_volume_attention.pkl
 - features_03_abnormal_sentiment.pkl


In [24]:
df = pd.read_pickle(feature_files[2])
df.head()

,symbol,date,n_bullish,n_bearish,total_labeled,bullish_ratio,bearish_ratio,net_sentiment,extreme_bullish_80,extreme_bullish_90,extreme_bearish_80,extreme_bearish_90,disagreement_index,abnormal_sentiment_1d,abnormal_sentiment_5d,abnormal_sentiment_21d,abnormal_sentiment_63d,abnormal_sentiment_250d
0,A,2010-06-02,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
1,A,2010-06-03,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
2,A,2010-06-04,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
3,A,2010-06-07,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
4,A,2010-06-08,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0


In [25]:
df.duplicated(['symbol', 'date']).sum()

0

In [26]:
df.head()

,symbol,date,n_bullish,n_bearish,total_labeled,bullish_ratio,bearish_ratio,net_sentiment,extreme_bullish_80,extreme_bullish_90,extreme_bearish_80,extreme_bearish_90,disagreement_index,abnormal_sentiment_1d,abnormal_sentiment_5d,abnormal_sentiment_21d,abnormal_sentiment_63d,abnormal_sentiment_250d
0,A,2010-06-02,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
1,A,2010-06-03,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
2,A,2010-06-04,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
3,A,2010-06-07,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
4,A,2010-06-08,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0


In [27]:
df.tail()

,symbol,date,n_bullish,n_bearish,total_labeled,bullish_ratio,bearish_ratio,net_sentiment,extreme_bullish_80,extreme_bullish_90,extreme_bearish_80,extreme_bearish_90,disagreement_index,abnormal_sentiment_1d,abnormal_sentiment_5d,abnormal_sentiment_21d,abnormal_sentiment_63d,abnormal_sentiment_250d
28206140,ZZ,2023-12-27,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
28206141,ZZ,2023-12-28,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
28206142,ZZ,2023-12-29,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
28206143,ZZ,2024-01-02,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
28206144,ZZ,2024-01-03,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0


In [28]:
df.reset_index().duplicated('index').sum()

0

In [29]:
df.reset_index()['index'].describe()

count    2.820614e+07
mean     1.410307e+07
std      8.142413e+06
min      0.000000e+00
25%      7.051536e+06
50%      1.410307e+07
75%      2.115461e+07
max      2.820614e+07
Name: index, dtype: float64

In [30]:
# Analyze each file
stats = []

for file_path in feature_files:
    print(f"Analyzing {file_path.name}...")
    try:
        df = pd.read_pickle(file_path)
        
        # Check primary key uniqueness
        is_unique = False
        if 'symbol' in df.columns and 'date' in df.columns:
            is_unique = not df.duplicated(subset=['symbol', 'date']).any()
        
        stats.append({
            'filename': file_path.name,
            'rows': len(df),
            'cols': len(df.columns),
            'unique_symbols': df['symbol'].nunique() if 'symbol' in df.columns else 0,
            'unique_dates': df['date'].nunique() if 'date' in df.columns else 0,
            'is_pk_unique': is_unique,
            'columns': list(df.columns)
        })
    except Exception as e:
        print(f"Error reading {file_path.name}: {e}")

# Create summary dataframe
df_stats = pd.DataFrame(stats)
display(df_stats[['filename', 'rows', 'cols', 'unique_symbols', 'unique_dates', 'is_pk_unique']])


Analyzing features_01_basic_sentiment.pkl...
Analyzing features_02_volume_attention.pkl...
Analyzing features_03_abnormal_sentiment.pkl...


,filename,rows,cols,unique_symbols,unique_dates,is_pk_unique
0,features_01_basic_sentiment.pkl,3505817,13,8245,3347,True
1,features_02_volume_attention.pkl,14193826,22,8245,3421,True
2,features_03_abnormal_sentiment.pkl,28206145,18,8245,3421,True


In [11]:
df.tail()

,symbol,date,n_bullish,n_bearish,total_labeled,bullish_ratio,bearish_ratio,net_sentiment,extreme_bullish_80,extreme_bullish_90,extreme_bearish_80,extreme_bearish_90,disagreement_index,abnormal_sentiment_1d,abnormal_sentiment_5d,abnormal_sentiment_21d,abnormal_sentiment_63d,abnormal_sentiment_250d
9274326,ZZ,2023-12-27,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
9274327,ZZ,2023-12-28,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
9274328,ZZ,2023-12-29,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
9274329,ZZ,2024-01-02,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
9274330,ZZ,2024-01-03,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0


In [31]:
# Investigate files with non-unique primary keys
non_unique_files = df_stats[~df_stats['is_pk_unique']]['filename'].tolist()

if non_unique_files:
    print(f"Files with non-unique (symbol, date): {non_unique_files}")
    
    for fname in non_unique_files:
        print(f"\nInspecting duplicates in {fname}:")
        file_path = FEATURES_DIR / fname
        df = pd.read_pickle(file_path)
        
        duplicates = df[df.duplicated(subset=['symbol', 'date'], keep=False)].sort_values(['symbol', 'date'])
        print(f"Number of duplicate rows: {len(duplicates)}")
        display(duplicates.head(10))
else:
    print("All files have unique (symbol, date) combinations.")


All files have unique (symbol, date) combinations.


In [32]:
# Compare symbol/date coverage
if not df_stats.empty:
    # Use the file with the most rows as reference
    ref_file = df_stats.loc[df_stats['rows'].idxmax(), 'filename']
    print(f"Reference file (max rows): {ref_file}")
    
    df_ref = pd.read_pickle(FEATURES_DIR / ref_file)
    ref_symbols = set(df_ref['symbol'].unique())
    ref_dates = set(df_ref['date'].unique())
    
    print("\nCoverage Comparison:")
    for fname in feature_files:
        if fname.name == ref_file:
            continue
            
        df_curr = pd.read_pickle(fname)
        curr_symbols = set(df_curr['symbol'].unique())
        curr_dates = set(df_curr['date'].unique())
        
        missing_symbols = len(ref_symbols - curr_symbols)
        extra_symbols = len(curr_symbols - ref_symbols)
        missing_dates = len(ref_dates - curr_dates)
        extra_dates = len(curr_dates - ref_dates)
        
        print(f"\n{fname.name}:")
        print(f"  Missing symbols vs ref: {missing_symbols}")
        print(f"  Extra symbols vs ref: {extra_symbols}")
        print(f"  Missing dates vs ref: {missing_dates}")
        print(f"  Extra dates vs ref: {extra_dates}")


Reference file (max rows): features_03_abnormal_sentiment.pkl

Coverage Comparison:

features_01_basic_sentiment.pkl:
  Missing symbols vs ref: 0
  Extra symbols vs ref: 0
  Missing dates vs ref: 74
  Extra dates vs ref: 0

features_02_volume_attention.pkl:
  Missing symbols vs ref: 0
  Extra symbols vs ref: 0
  Missing dates vs ref: 0
  Extra dates vs ref: 0


In [33]:
# Compare (Symbol, Date) Combinations
print("Comparing (Symbol, Date) combinations across files...")

# Load all (symbol, date) pairs
file_pairs = {}
for f in feature_files:
    print(f"Loading keys from {f.name}...")
    df = pd.read_pickle(f)
    # Create a set of tuples
    pairs = set(zip(df['symbol'], df['date']))
    file_pairs[f.name] = pairs
    print(f"  Count: {len(pairs):,}")

# Calculate intersection (common to all)
common_pairs = set.intersection(*file_pairs.values())
print(f"\nCommon (Symbol, Date) pairs across ALL files: {len(common_pairs):,}")

# Calculate union (total unique pairs)
all_pairs = set.union(*file_pairs.values())
print(f"Total unique (Symbol, Date) pairs across ANY file: {len(all_pairs):,}")

# Pairwise comparison
print("\nPairwise Overlap:")
filenames = list(file_pairs.keys())
for i in range(len(filenames)):
    for j in range(i+1, len(filenames)):
        f1 = filenames[i]
        f2 = filenames[j]
        
        s1 = file_pairs[f1]
        s2 = file_pairs[f2]
        
        overlap = len(s1.intersection(s2))
        print(f"  {f1} vs {f2}:")
        print(f"    Overlap: {overlap:,}")
        print(f"    Unique to {f1}: {len(s1 - s2):,}")
        print(f"    Unique to {f2}: {len(s2 - s1):,}")


Comparing (Symbol, Date) combinations across files...
Loading keys from features_01_basic_sentiment.pkl...
  Count: 3,505,817
Loading keys from features_02_volume_attention.pkl...
  Count: 14,193,826
Loading keys from features_03_abnormal_sentiment.pkl...
  Count: 28,206,145

Common (Symbol, Date) pairs across ALL files: 3,505,817
Total unique (Symbol, Date) pairs across ANY file: 28,206,145

Pairwise Overlap:
  features_01_basic_sentiment.pkl vs features_02_volume_attention.pkl:
    Overlap: 3,505,817
    Unique to features_01_basic_sentiment.pkl: 0
    Unique to features_02_volume_attention.pkl: 10,688,009
  features_01_basic_sentiment.pkl vs features_03_abnormal_sentiment.pkl:
    Overlap: 3,505,817
    Unique to features_01_basic_sentiment.pkl: 0
    Unique to features_03_abnormal_sentiment.pkl: 24,700,328
  features_02_volume_attention.pkl vs features_03_abnormal_sentiment.pkl:
    Overlap: 14,193,826
    Unique to features_02_volume_attention.pkl: 0
    Unique to features_03_abno